In [2]:
import sys
sys.path.append("..")

In [2]:
from sqlalchemy import text

In [3]:
from infrastructure.dependencies.vector_db import VertexVectorDBClient
from app.services.indexer import process_products, generate_image_embedding
from infrastructure.database import SessionLocal
from config.config import settings

from infrastructure.dependencies.vector_db import get_client, get_existing_ids
from config.globals import MODELS
from app.services.search import SearchService
import numpy as np

/Users/alvin/Documents/GitHub/test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
MODELS['search'] = SearchService()

In [5]:
client = get_client()
db = SessionLocal()
collection_name = settings.VERTEX_INDEX_ENDPOINT

In [18]:
db.rollback()

query = text("SELECT * FROM products LIMIT 10")
result = db.execute(query)

result = result.mappings().fetchall()

candidate_ids = [row['id'] for row in result]

unsaved_ids, saved_ids = get_existing_ids(client, candidate_ids)

In [13]:
len(candidate_ids)

1046

In [15]:
len(saved_ids)

1046

In [16]:
unsaved_ids

set()

In [19]:
product_count = 0
indexed_count = 0

for row in result:
    product_count += 1
    product_id = row.get('id')
    main_image_url = row.get('main_image')

    embedding = generate_image_embedding(main_image_url)
    embedding_list = np.array(embedding.detach().numpy()).flatten().tolist()
    point = {
        "id": product_id,
        "vector": embedding_list,
    }

In [23]:
print(main_image_url)
print(len(embedding_list))

https://storage.googleapis.com/images_modalab/designers/A%20modo%20mio%20/Tea%20Jacket%20/A_Modo_Mio-Tea_Jacket-Photo_Product-Gray_Blue.webp
512


In [54]:
text_embedding = await MODELS['search'].generate_text_embedding("red dress")

In [ ]:
from google.cloud import aiplatform_v1
from config import settings
from types import SimpleNamespace

INDEX_ENDPOINT = settings.VERTEX_INDEX_ENDPOINT
DEPLOYED_INDEX_ID = settings.VERTEX_DEPLOYED_INDEX_ID
API_ENDPOINT = settings.VERTEX_API_ENDPOINT


match_client_options = {"api_endpoint": API_ENDPOINT}
match_client = aiplatform_v1.MatchServiceClient(
    client_options=match_client_options or None
)


query_vector = text_embedding
datapoint = aiplatform_v1.IndexDatapoint(feature_vector=query_vector)
query = aiplatform_v1.FindNeighborsRequest.Query(
    datapoint=datapoint,
    neighbor_count=5,
)
request = aiplatform_v1.FindNeighborsRequest(
    index_endpoint=INDEX_ENDPOINT,
    deployed_index_id=DEPLOYED_INDEX_ID,
    queries=[query],
    return_full_datapoint=True,
)

response = match_client.find_neighbors(request=request)
neighbors = response.nearest_neighbors[0].neighbors
results = []
for neighbor in neighbors:
    id_ = neighbor.datapoint.datapoint_id
    results.append(SimpleNamespace(payload=int(id_)))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [57]:
results_list = [r.payload for r in results]

db.rollback()
query = text(f"SELECT id, main_image FROM products WHERE id IN {tuple(results_list)}")
comparison = db.execute(query)
result = comparison.mappings().fetchall()

In [ ]:
result

[{'id': 1736, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Palomiino/Spring%20Summer%2025/Gae%20Top/palomiino-gae_top-red-photo_product.webp'},
 {'id': 1744, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Palomiino/Spring%20Summer%2025/Wes%20Leggings/palomiino-wes_leggings-red-photo_product.webp'},
 {'id': 1936, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Saudade%20/Scarlet%20Top/saudade_da_voce-scarlet-top-photo-product-main-image.webp'},
 {'id': 1937, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Saudade%20/Scarlet%20Skirt/saudade_da_voce-scarlet-skirt-photo-product-main-image.webp'},
 {'id': 2278, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Mola%20Mola/Calypso%20Bianca%20One%20Piece/Mola_Mola-Calypso_Bianca_One_Piece-Photo_Product.webp'}]

# Second Test

In [1]:
import sys
sys.path.append("..")

In [2]:
from app.services.search import SearchService
from app.schemas import SearchByDescription
from infrastructure.dependencies.vector_db import get_client
from infrastructure.database import SessionLocal
from sqlalchemy import text

/Users/alvin/Documents/GitHub/test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connecting to Cloud SQL instance: modalab-435501:us-east1:modalab-db-postgres


In [3]:
client = get_client()
db = SessionLocal()

In [4]:
search_service = SearchService()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [21]:
request = SearchByDescription(text="red", top_k=5, followed_stores=[87])
result = await search_service.search(request)
result_list = [p.id for p in result.products]

In [22]:
db.rollback()
query = text(f"SELECT id, designer_id, main_image FROM products WHERE id IN {tuple(result_list)}")
comparison = db.execute(query)
result = comparison.mappings().fetchall()
result

[{'id': 2618, 'designer_id': 87, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Project%20Adamo%20/Positano%20Dress%20/project_adamo-positano_dress-ivory-photo_product.webp'},
 {'id': 2620, 'designer_id': 87, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Project%20Adamo%20/etereo%20top%20/etereo%20top%20ivory%20/project_adamo-etereo_top-ivory-photo_product.webp'},
 {'id': 2629, 'designer_id': 87, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Project%20Adamo%20/tomatini%20t-shirt/project_adamo-tomatini_t_shirt-ivory-photo_product.webp'},
 {'id': 2655, 'designer_id': 87, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Project%20Adamo%20/rosso%20dress/project_adamo-rosso_dress-red-photo_product.webp'},
 {'id': 2656, 'designer_id': 87, 'main_image': 'https://storage.googleapis.com/images_modalab/designers/Project%20Adamo%20/pomodoro%20dress/project_adamo-pomodoro_dress-red-photo_product.webp'}]